# TUIA NLP 2025 - TRABAJO PRÁCTICO FINAL - Agosto 2025


## *Contexto Temático: El Universo Cinematográfico "Cosmos Nexus"*


## Alumna: Julia Sumiacher | Legajo: S-57932



---



## Configuración del entorno

### Instalación de dependencias

In [ ]:
!pip install sqlalchemy chromadb sentence-transformers rank-bm25 pdfplumber neo4j pandas pypdf transformers accelerate sentencepiece bitsandbytes --quiet -q "chromadb>=0.5.4" "sentence-transformers>=3.0.0" "tiktoken>=0.7.0" "pypdf>=4.2.0" "tqdm>=4.66.0" duckduckgo-search==6.* wikipedia --quiet langchain==0.2.17 langchain-community==0.2.17

### Imports

In [ ]:
import os
import sqlite3
import pdfplumber
import pandas as pd
import sqlite3, shutil
import zipfile
import chromadb
import requests
import gc
import time
import torch
import tiktoken
import numpy as np
import wikipedia
import urllib.parse
import re, io, zipfile, hashlib, unicodedata, json, math
from langchain.tools import tool
from google.colab import userdata
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import Tool as LCTool
from pprint import pprint
from langchain_core.tools import BaseTool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from urllib.parse import quote_plus
from duckduckgo_search import DDGS
from textwrap import shorten
from openai import OpenAI
from pathlib import Path
from typing import List, Dict, Optional, Iterable, Tuple, Protocol
from tqdm import tqdm
from chromadb.utils import embedding_functions
from pypdf import PdfReader
from pathlib import Path
from neo4j import GraphDatabase
from sqlalchemy import create_engine, text
from huggingface_hub import InferenceClient
from sentence_transformers import SentenceTransformer
from chromadb import Client
from chromadb.config import Settings
from rank_bm25 import BM25Okapi
from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple
import os, re, json, random, textwrap
from typing import Optional
from transformers import AutoTokenizer
from sentence_transformers import CrossEncoder

## Ejercicio 1: Diseño e Implementación de un Chatbot Multi-Fuente


### Construcción de la Base de Conocimiento

#### Configuración de rutas

In [ ]:
zip_path = "/content/corpus_conocimiento.zip"
extract_dir = "/content/corpus_conocimiento"

os.makedirs(extract_dir, exist_ok=True)

try:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extracción completada con éxito: {zip_path} en {extract_dir}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo ZIP en {zip_path}")
except zipfile.BadZipFile:
    print(f"Error: El archivo en {zip_path} no es un archivo ZIP válido")
except Exception as e:
    print(f"Ocurrió un error durante la extracción: {e}")

BASE_DIR = Path("/content")

# Archivos
SQLITE_DB_SRC = BASE_DIR / "cosmos_nexus_db.db"
GRAFO_CSV      = BASE_DIR / "relaciones_universo.csv"

# Salidas
SQLITE_DB_PATH = BASE_DIR / "cosmos_nexus.db"
CHROMA_DIR     = BASE_DIR / "chroma_cosmos"
CORPUS_DIR     = BASE_DIR / "corpus_conocimiento"

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

#### Base de Datos SQL

In [ ]:
# Limpiar DB de trabajo anterior si existía
if SQLITE_DB_PATH.exists():
    SQLITE_DB_PATH.unlink()

if SQLITE_DB_SRC.exists():
    shutil.copy2(SQLITE_DB_SRC, SQLITE_DB_PATH)
    print(f"Usando la DB provista: {SQLITE_DB_SRC} -> {SQLITE_DB_PATH}")
else:
    raise FileNotFoundError("No se encontró 'cosmos_nexus_db.db' en /content.")

# Inspección rápida
conn = sqlite3.connect(str(SQLITE_DB_PATH))
tables = [t[0] for t in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()]
print("Tablas en la DB:")
for t in tables:
    print(" -", t)
    try:
        cols = [row[1] for row in conn.execute(f"PRAGMA table_info({t});").fetchall()]
        sample = conn.execute(f"SELECT * FROM {t} LIMIT 3;").fetchall()
        print("   Columnas:", cols)
        for r in sample:
            print("   ", r)
    except Exception as e:
        print("   (No se pudo mostrar muestra):", e)
conn.close()
print(f"\nSQLite lista en {SQLITE_DB_PATH}")


#### Base de Datos de Grafos

##### Configuraciones básicas

In [ ]:
NEO4J_VERSION = "5.22.0"
NEO4J_HOME = "/content/neo4j"
NEO4J_TGZ = "/content/neo4j.tgz"
INITIAL_PASSWORD = "neo4j1234"  # se puede cambiar

# Instalar Java 17
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jre-headless > /dev/null

# Descargar Neo4j Community
!wget -q https://dist.neo4j.org/neo4j-community-{NEO4J_VERSION}-unix.tar.gz -O {NEO4J_TGZ}

# Descomprimir en /content/neo4j
!mkdir -p {NEO4J_HOME}
!tar -xzf {NEO4J_TGZ} -C {NEO4J_HOME} --strip-components=1

# Establecer password inicial (solo antes del primer arranque)
!{NEO4J_HOME}/bin/neo4j-admin dbms set-initial-password {INITIAL_PASSWORD}

print("Neo4j instalado en", NEO4J_HOME)

##### Arrancar Neo4j

In [ ]:
NEO4J_HOME = "/content/neo4j"
INITIAL_PASSWORD = "neo4j1234"

# Arrancar el servidor (daemon)
!{NEO4J_HOME}/bin/neo4j start

# Esperar unos segundos a que levante
time.sleep(10)

# Estado del servicio
!{NEO4J_HOME}/bin/neo4j status

# Probar la conexión con cypher-shell
!{NEO4J_HOME}/bin/cypher-shell -a bolt://localhost:7687 -u neo4j -p {INITIAL_PASSWORD} "RETURN 1 AS ok;"

##### Normalizar CSV

In [ ]:
CSV_PATH = BASE_DIR / "relaciones_universo.csv"
NEO4J_HOME = "/content/neo4j"
OUT_PATH = f"{NEO4J_HOME}/import/relaciones_universo_norm.csv"

assert os.path.exists(CSV_PATH), f"No encuentro el archivo en {CSV_PATH}. Ajusta CSV_PATH."

# Leemos con tolerancia a BOM y comillas
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

# Detectar columnas: soporta 'entidad1, relacion, entidad2' y otras variantes
cols_lower = [c.lower() for c in df.columns]
def pick_first(cands):
    for c in cands:
        if c in cols_lower:
            return df.columns[cols_lower.index(c)]
    return None

src_col = pick_first(["entidad1","src","source","origen","from","desde"])
rel_col = pick_first(["relacion","rel","relation","tipo_relacion","edge","link"])
dst_col = pick_first(["entidad2","dst","target","destino","to","hacia"])

required = {"src": src_col, "rel": rel_col, "dst": dst_col}
if any(v is None for v in required.values()):
    raise ValueError(
        f"No se pudieron mapear columnas mínimas (src, rel, dst) desde {list(df.columns)}. "
        f"Detectado: {required}"
    )

# Normalizadores auxiliares
def remove_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if unicodedata.category(ch) != 'Mn')

def normalize_token(s: str) -> str:
    s = str(s).strip()
    s = remove_accents(s)
    s = s.upper()
    s = re.sub(r"[^\w\s/]", "_", s)  # sustituye signos por _
    s = s.replace(" ", "_").replace("-", "_")
    s = re.sub(r"_+", "_", s)        # colapsa múltiples _
    return s

# Aliasado amplio basado en tu CSV (unifica variantes y acentos)
ALIASES = {
    # identidad / familia
    "ES_HERMANO_DE": "ES_HERMANO_DE",
    "HERMANO_DE": "ES_HERMANO_DE",

    "ES_IDENTIDAD_DE": "ES_IDENTIDAD_DE",
    "IDENTIDAD_DE": "ES_IDENTIDAD_DE",

    # pertenencia y liderazgo
    "ES_MIEMBRO_DE": "ES_MIEMBRO_DE",
    "MIEMBRO_DE": "ES_MIEMBRO_DE",
    "TRABAJA_CON": "TRABAJA_CON",
    "TRABAJO_CON": "TRABAJA_CON",
    "TRABAJA_PARA": "TRABAJA_PARA",
    "TRABAJO_PARA": "TRABAJA_PARA",

    "ES_LIDER_DE": "ES_LIDER_DE",
    "LIDERA": "ES_LIDER_DE",
    "COMANDA": "ES_LIDER_DE",

    # aliados / enemigos / conflicto
    "ES_ALIADO_DE": "ES_ALIADO_DE",
    "ES_ALIADA_DE": "ES_ALIADO_DE",
    "ALIADO_DE": "ES_ALIADO_DE",
    "ALIADA_DE": "ES_ALIADO_DE",
    "ALLY": "ES_ALIADO_DE",
    "ALLY_OF": "ES_ALIADO_DE",

    "ES_ENEMIGO_DE": "ES_ENEMIGO_DE",
    "ES_ENEMIGA_DE": "ES_ENEMIGO_DE",
    "ENEMIGO_DE": "ES_ENEMIGO_DE",
    "ENEMIGA_DE": "ES_ENEMIGO_DE",
    "SE_ENFRENTA_A": "SE_ENFRENTA_A",
    "COMBATIO_CONTRA": "COMBATIO_CONTRA",
    "TRAICIONO_A": "TRAICIONO_A",

    # mentoría / entrenamiento / consejo
    "ES_MENTOR_DE": "ES_MENTOR_DE",
    "MENTOR_DE": "ES_MENTOR_DE",
    "ENTRENO_A": "ENTRENO_A",
    "ES_CONSEJERA_DE": "ES_CONSEJERA_DE",
    "CONSEJERA_DE": "ES_CONSEJERA_DE",
    "CONSEJERO_DE": "ES_CONSEJERA_DE",

    # apariciones en obras y rol narrativo
    "APARECE_EN": "APARECE_EN",
    "ES_VILLANO_EN": "ES_VILLANO_EN",
    "PROTAGONIZA": "PROTAGONIZA",
    "DIRIGIDA_POR": "DIRIGIDA_POR",
    "DIRIGIDO_POR": "DIRIGIDA_POR",

    # procedencia / localización / actividad
    "PROVIENE_DE": "PROVIENE_DE",
    "NACIO_EN": "NACIO_EN",
    "VIVE_EN": "VIVE_EN",
    "OPERA_DESDE": "OPERA_DESDE",
    "TIENE_BASE_EN": "TIENE_BASE_EN",
    "EXILIADA_DE": "EXILIADA_DE",
    "EXILIADO_DE": "EXILIADA_DE",
    "FRECUENTA": "FRECUENTA",
    "ESTUDIA_EN": "ESTUDIA_EN",
    "CUSTODIA": "CUSTODIA",
    "OPERA": "OPERA",

    # objetos / poderes / acciones
    "USA": "USA",
    "POSEE": "POSEE",
    "TIENE": "POSEE",            # unificamos TIENE -> POSEE
    "MANIPULA": "MANIPULA",
    "EMPUÑA": "EMPUÑA",
    "PERDIO": "PERDIO",
    "VISTE": "VISTE",
    "CONTROLA": "CONTROLA",
    "CONTROLAN": "CONTROLA",
    "PORTA": "PORTA",
    "CREO": "CREO",
    "MANTIENE": "MANTIENE",
    "BUSCA": "BUSCA",
    "COMUNICA_A_TRAVES": "COMUNICA_A_TRAVES",
    "DISTORSIONA": "DISTORSIONA",
    "DESTRUYO": "DESTRUYO",

    # relaciones de protección/defensa
    "PROTEGIDO_POR": "PROTEGIDO_POR",
    "PROTEGE": "PROTEGE",
    "DEFIENDE": "DEFIENDE",
    "AMENAZA": "AMENAZA",

    # cadenas temporales entre obras
    "PRECEDE_A": "PRECEDE_A",
    "ES_PRECUELA_DE": "PRECEDE_A",

    # recursos / producción / suministro
    "PROVEE": "PROVEE",
    "ERA_FUENTE_DE": "ES_FUENTE_DE",
    "ES_FUENTE_DE": "ES_FUENTE_DE",
    "PRODUCE": "PRODUCE",
    "SUMINISTRA": "SUMINISTRA",
    "GENERA": "GENERA",
    "CONTIENTE": "CONTIENE",  # por si llega mal tipeado
    "CONTIENE": "CONTIENE",
    "FACILITA": "FACILITA",

    # capacidad / predicción
    "PUEDE_DESTRUIR": "PUEDE_DESTRUIR",
    "PREDICE": "PREDICE",
}

USE_ALIASES = True  # Poner en False para conservar EXACTAMENTE el valor del CSV

def normalize_relation(r):
    tok = normalize_token(r)
    if USE_ALIASES:
        return ALIASES.get(tok, tok)
    return tok

# Construimos el dataframe normalizado
ndf = pd.DataFrame({
    "src": df[src_col].astype(str),
    "rel": df[rel_col].astype(str).map(normalize_relation),
    "dst": df[dst_col].astype(str),
    # etiquetas genéricas
    "src_label": "Entidad",
    "dst_label": "Entidad",
})

# Limpiar filas vacías
ndf = ndf.dropna(subset=["src","rel","dst"])
ndf = ndf[(ndf["src"].str.strip()!="") &
          (ndf["rel"].str.strip()!="") &
          (ndf["dst"].str.strip()!="")]

# Guardar en import
os.makedirs(f"{NEO4J_HOME}/import", exist_ok=True)
ndf.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("CSV normalizado guardado en:", OUT_PATH)
ndf.head(10)


##### Cargar datos con LOAD CSV

In [ ]:
NEO4J_HOME = "/content/neo4j"
PASSWORD = "neo4j1234"

# Constraint en una consulta separada (es normal que diga 0 rows)
constraint_q = r"""
CREATE CONSTRAINT entidad_name IF NOT EXISTS
FOR (n:Entidad) REQUIRE n.name IS UNIQUE;
"""
with open("/content/create_constraint.cypher", "w", encoding="utf-8") as f:
    f.write(constraint_q)

print("Creando constraint...")
!{NEO4J_HOME}/bin/cypher-shell -a bolt://localhost:7687 -u neo4j -p {PASSWORD} -f /content/create_constraint.cypher

# Carga masiva con CALL IN TRANSACTIONS (sintaxis moderna)
load_q = r"""
CALL {
  LOAD CSV WITH HEADERS FROM 'file:///relaciones_universo_norm.csv' AS row
  WITH row
  WHERE row.src IS NOT NULL AND trim(row.src) <> ''
    AND row.dst IS NOT NULL AND trim(row.dst) <> ''
    AND row.rel IS NOT NULL AND trim(row.rel) <> ''
  MERGE (s:Entidad {name: row.src})
    ON CREATE SET s.tipo = coalesce(row.src_label, 'Entidad')
  MERGE (t:Entidad {name: row.dst})
    ON CREATE SET t.tipo = coalesce(row.dst_label, 'Entidad')
  MERGE (s)-[r:RELACION]->(t)
    ON CREATE SET r.tipo = row.rel
    ON MATCH SET  r.tipo = coalesce(r.tipo, row.rel)
} IN TRANSACTIONS OF 1000 ROWS;
"""

with open("/content/load_csv_modern.cypher", "w", encoding="utf-8") as f:
    f.write(load_q)

print("\nCargando datos con CALL IN TRANSACTIONS...")
!{NEO4J_HOME}/bin/cypher-shell -a bolt://localhost:7687 -u neo4j -p {PASSWORD} -f /content/load_csv_modern.cypher

# Verificar que los datos se cargaron correctamente
verify_q = r"""
MATCH (n:Entidad)
RETURN count(n) as total_entidades;
"""

with open("/content/verify_load.cypher", "w", encoding="utf-8") as f:
    f.write(verify_q)

print("\nVerificando carga...")
!{NEO4J_HOME}/bin/cypher-shell -a bolt://localhost:7687 -u neo4j -p {PASSWORD} -f /content/verify_load.cypher

# Ver algunos ejemplos de relaciones
sample_q = r"""
MATCH (s:Entidad)-[r:RELACION]->(t:Entidad)
RETURN s.name, r.tipo, t.name
LIMIT 5;
"""

with open("/content/sample_relations.cypher", "w", encoding="utf-8") as f:
    f.write(sample_q)

print("\nMostrando ejemplos de relaciones...")
!{NEO4J_HOME}/bin/cypher-shell -a bolt://localhost:7687 -u neo4j -p {PASSWORD} -f /content/sample_relations.cypher

#### Base de Datos Vectorial

##### Configuración Básica

In [ ]:
# Rutas base
BASE_DIR = Path.cwd()
CHROMA_DIR = BASE_DIR / "chroma_db"
CORPUS_DIR = BASE_DIR / "corpus_conocimiento"

# Parámetros de chunking
CHUNK_TOKENS = 300
OVERLAP_TOKENS = 60

# Modelo de embeddings
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Colección en ChromaDB
CHROMA_COLLECTION_NAME = "corpus_cosmos_nexus"

##### Lectura de archivos

In [ ]:
def normalize_text(s: str) -> str:
    """Normaliza unicode, colapsa espacios y quita espacios repetidos."""
    s = unicodedata.normalize("NFKC", s)
    # Reemplazar saltos de línea múltiples por uno solo
    s = re.sub(r"\r\n|\r", "\n", s)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)  # máximo 2 saltos consecutivos
    return s.strip()

def read_txt_file(path: Path) -> List[Tuple[Optional[int], str]]:
    """Devuelve lista [(None, texto_completo)] para TXT."""
    text = path.read_text(encoding="utf-8", errors="ignore")
    return [(None, normalize_text(text))]

def read_pdf_file(path: Path) -> List[Tuple[int, str]]:
    """
    Devuelve lista [(page_idx, texto_pagina)] para PDF.
    page_idx es 0-based.
    """
    pages = []
    with open(path, "rb") as f:
        reader = PdfReader(f)
        for i, page in enumerate(reader.pages):
            try:
                txt = page.extract_text() or ""
            except Exception:
                txt = ""
            pages.append((i, normalize_text(txt)))
    return pages

def iter_documents(corpus_dir: Path) -> Iterable[Dict]:
    """
    Itera documentos del corpus y rinde dicts con:
    {
      'source': str (ruta relativa),
      'page': Optional[int],
      'text': str
    }
    - Para TXT: page=None, 1 item por archivo.
    - Para PDF: page=índice de página, 1 item por página.
    """
    assert corpus_dir.exists(), f"No existe {corpus_dir}"
    for path in sorted(corpus_dir.rglob("*")):
        if path.is_dir():
            continue
        rel = str(path.relative_to(corpus_dir))
        if path.suffix.lower() == ".txt":
            for page, text in read_txt_file(path):
                if text:
                    yield {"source": rel, "page": page, "text": text}
        elif path.suffix.lower() == ".pdf":
            for page, text in read_pdf_file(path):
                if text:
                    yield {"source": rel, "page": page, "text": text}
        # Si hay otros formatos, se podrían agregar aquí (md, html, etc.)

# Prueba rápida de lectura (muestra 1-2 ejemplos)
if CORPUS_DIR.exists():
    print("Muestra de lectura de documentos:")
    count = 0
    for doc in iter_documents(CORPUS_DIR):
        print(f" • {doc['source']} | page={doc['page']} | len={len(doc['text'])}")
        print(doc["text"][:250].replace("\n"," ") + ("..." if len(doc["text"])>250 else ""))
        count += 1
        if count >= 2:
            break

##### Tokenización y chunking

In [ ]:
# Usamos el tokenizer 'cl100k_base' (el de los modelos GPT-4/3.5-turbo).
# Es robusto y estable para conteo aproximado de tokens.
TK = tiktoken.get_encoding("cl100k_base")

def tokens_of(text: str) -> List[int]:
    """
    Toma un texto y lo convierte en una lista de tokens numéricos
    utilizando el codificador `TK`.
    """
    return TK.encode(text, allowed_special=set())


def count_tokens(text: str) -> int:
    """
    Calcula cuántos tokens hay en un texto
    utilizando la función `tokens_of`.
    """
    return len(tokens_of(text))


def chunk_text(
    text: str,
    chunk_tokens: int = CHUNK_TOKENS,
    overlap_tokens: int = OVERLAP_TOKENS
) -> List[str]:
    """
    Divide 'text' en chunks por tokens, con superposición de 'overlap_tokens'.
    Implementación simple por ventana deslizante sobre tokens.
    """
    toks = tokens_of(text)
    chunks = []
    i = 0
    while i < len(toks):
        window = toks[i : i + chunk_tokens]
        if not window:
            break
        chunk = TK.decode(window)
        chunks.append(chunk.strip())
        if i + chunk_tokens >= len(toks):
            break
        i += (chunk_tokens - overlap_tokens)
    return chunks

def chunk_document(doc: Dict, chunk_tokens: int = CHUNK_TOKENS, overlap_tokens: int = OVERLAP_TOKENS) -> List[Dict]:
    """
    Toma un dict {'source', 'page', 'text'} y devuelve lista de chunks:
    [{
       'chunk_text': str,
       'source': str,
       'page': Optional[int],
       'chunk_idx': int
    }, ...]
    """
    parts = chunk_text(doc["text"], chunk_tokens=chunk_tokens, overlap_tokens=overlap_tokens)
    out = []
    for i, part in enumerate(parts):
        out.append({
            "chunk_text": part,
            "source": doc["source"],
            "page": doc["page"],
            "chunk_idx": i
        })
    return out

# Prueba: chunkear el primer documento encontrado
if CORPUS_DIR.exists():
    for d in iter_documents(CORPUS_DIR):
        chunks = chunk_document(d)
        print(f"Documento: {d['source']} page={d['page']}")
        print(f" - Tokens totales: {count_tokens(d['text'])}")
        print(f" - Chunks generados: {len(chunks)} | Ejemplo (primer chunk):")
        if chunks:
            print(chunks[0]["chunk_text"][:300].replace("\n"," ") + ("..." if len(chunks[0]["chunk_text"])>300 else ""))
        break

##### Modelo de embeddings

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

emb_model = SentenceTransformer(EMB_MODEL_NAME, device=device)
print(f"Modelo de embeddings cargado: {EMB_MODEL_NAME}")

def embed_texts(texts: List[str], batch_size: int = 256, normalize: bool = True) -> np.ndarray:
    """
    Calcula embeddings para una lista de textos.
    Retorna un array shape [N, D].
    """
    embs = emb_model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
        show_progress_bar=True
    )
    return embs

##### Inicializar ChromaDB

In [ ]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Nota: no se confugura embedding_function en la colección para poder
# controlar explícitamente qué embeddings se guardan (los que generamos arriba).
try:
    collection = client.get_collection(CHROMA_COLLECTION_NAME)
    print(f"Se reutiliza colección existente: '{CHROMA_COLLECTION_NAME}'")
except Exception:
    collection = client.create_collection(
        name=CHROMA_COLLECTION_NAME,
        metadata={"description": "Vector DB de 'corpus_conocimiento' (Cosmos Nexus) con embeddings ST."}
    )
    print(f"Colección creada: '{CHROMA_COLLECTION_NAME}'")


##### Función de ingesta

In [ ]:
def make_id(source: str, page: Optional[int], chunk_idx: int, text: str) -> str:
    """
    Crea un ID único y estable por chunk usando hash del contenido + metadatos.
    """
    base = f"{source}|p{page if page is not None else 'NA'}|c{chunk_idx}|{text[:64]}"
    h = hashlib.sha1(base.encode("utf-8", errors="ignore")).hexdigest()[:16]
    return f"{source}#p{page if page is not None else 'NA'}#c{chunk_idx}-{h}"

def ingest_corpus(
    corpus_dir: Path,
    collection,
    chunk_tokens: int = CHUNK_TOKENS,
    overlap_tokens: int = OVERLAP_TOKENS,
    embed_batch_size: int = 256,
    add_batch_size: int = 512
) -> Dict:
    """
    Pipeline completo:
      - Lee documentos del corpus
      - Chunking token-aware
      - Embeddings con SentenceTransformer
      - Inserción en Chroma en batches
    """
    assert corpus_dir.exists(), f"No existe {corpus_dir}"

    # 1) Recolectar todos los chunks con metadatos
    all_chunks: List[Dict] = []
    print("→ Leyendo y chunkendo documentos...")
    for doc in tqdm(iter_documents(corpus_dir), total=None):
        chs = chunk_document(doc, chunk_tokens=chunk_tokens, overlap_tokens=overlap_tokens)
        all_chunks.extend(chs)
    if not all_chunks:
        print("⚠️ No se generaron chunks. ¿Hay texto en el corpus?")
        return {"chunks": 0, "added": 0}

    # 2) Preparar lotes para embeddings e inserción
    print(f"→ Total de chunks a procesar: {len(all_chunks)}")
    texts = [c["chunk_text"] for c in all_chunks]
    sources = [c["source"] for c in all_chunks]
    pages = [c["page"] for c in all_chunks]
    chunk_idxs = [c["chunk_idx"] for c in all_chunks]
    ids = [make_id(s, p, idx, t) for s, p, idx, t in zip(sources, pages, chunk_idxs, texts)]
    metadatas = [{"source": s, "page": (p if p is not None else -1), "chunk_idx": idx}
                 for s, p, idx in zip(sources, pages, chunk_idxs)]

    # 3) Calcular embeddings en batch
    print("→ Calculando embeddings...")
    embeddings = embed_texts(texts, batch_size=embed_batch_size, normalize=True)  # shape [N, D]

    # 4) Insertar en Chroma por lotes
    print("→ Insertando en ChromaDB...")
    added = 0
    for start in tqdm(range(0, len(ids), add_batch_size)):
        end = start + add_batch_size
        collection.add(
            ids=ids[start:end],
            documents=texts[start:end],
            metadatas=metadatas[start:end],
            embeddings=embeddings[start:end].tolist()
        )
        added += (end - start)

    try:
        count = collection.count()
    except Exception:
        count = None

    print(f"Ingesta finalizada. Chunks añadidos en esta corrida: {added}")
    if count is not None:
        print(f"Total en colección '{collection.name}': {count} registros")
    return {"chunks": len(all_chunks), "added": added, "collection_count": count}

# Ejecutar ingesta (puede tardar unos minutos según el tamaño del corpus)
if CORPUS_DIR.exists():
    stats = ingest_corpus(CORPUS_DIR, collection)
    print("Resumen:", stats)
else:
    print("No existe CORPUS_DIR. Sube la carpeta/zip y vuelve a ejecutar.")

##### Prueba de búsqueda semántica

In [ ]:
# Nota: Como la colección no tiene embedding_function asociada, para query
# pasamos query_embeddings calculados con el mismo modelo de arriba.

def search_semantic(query: str, top_k: int = 5):
    """
    Realiza una búsqueda semántica en la colección de documentos
    usando embeddings del texto de la consulta y muestra los resultados
    con su fuente, página, fragmento y puntaje aproximado.
    """
    q_emb = embed_texts([query], batch_size=1, normalize=True)[0]
    res = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    # Mostrar resultados
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]
    print(f"\n🔎 Consulta: {query}\n")
    for i, (doc, md, dist) in enumerate(zip(docs, metas, dists), start=1):
        src = md.get("source")
        page = md.get("page")
        cidx = md.get("chunk_idx")
        snippet = doc[:220].replace("\n", " ") + ("..." if len(doc) > 220 else "")
        print(f"{i:02d}. score≈{1.0 - dist:.4f} | {src} | page={page} | chunk={cidx}\n    {snippet}\n")

# Ejemplos de prueba (ajusta según tu corpus)
try:
    search_semantic("¿Quién es Captain Stellar?")
    search_semantic("alianzas y enemistades entre personajes")
except Exception as e:
    print("Nota: Si no hay resultados, ajusta la consulta o verifica que el corpus tenga contenido relacionado.")
    print("Detalle:", e)

### Implementación del Enrutador de Intenciones

#### Configuración básica

In [ ]:
random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LABELS = ("SQL", "GRAFO", "VECTOR")

@dataclass
class RoutingResult:
    etiqueta: str               # "SQL" | "GRAFO" | "VECTOR"
    justificacion: str          # breve oración explicativa
    salida_cruda: Optional[str] = None  # texto tal como lo devolvió el LLM

def normaliza_etiqueta(txt: str) -> Optional[str]:
    """
    Normaliza una etiqueta de texto para identificar si corresponde
    a "SQL", "GRAFO" o "VECTOR", aceptando también variantes comunes
    o formatos alternativos dentro de frases o en JSON.
    """
    if not txt:
        return None
    t = txt.strip().upper()
    # Permitir variantes comunes
    mapa = {
        "SQL": "SQL",
        "BASE DE DATOS SQL": "SQL",
        "BD SQL": "SQL",
        "GRAFO": "GRAFO",
        "GRAPH": "GRAFO",
        "NEO4J": "GRAFO",
        "VECTOR": "VECTOR",
        "VECTORES": "VECTOR",
        "RAG": "VECTOR",
        "CHROMADB": "VECTOR"
    }
    # Detectar dentro de frases
    for k,v in mapa.items():
        if re.search(rf"\b{k}\b", t):
            return v
    # JSON {"label":"SQL"} o similar
    m = re.search(r'"label"\s*:\s*"(?P<lab>SQL|GRAFO|VECTOR)"', txt, re.I)
    if m:
        return m.group("lab").upper()
    # Etiqueta simple en la primera palabra
    m = re.match(r'\s*(SQL|GRAFO|VECTOR)\b', t)
    if m:
        return m.group(1).upper()
    return None

#### Prompt few-shot para el LLM

In [ ]:
FEW_SHOTS = [
    # --- SQL ---
    ("¿En qué año se estrenó 'Aegis Rising'?", "SQL",
     "Pide un atributo tabular (año de estreno) de una película concreta."),
    ("Lista las películas dirigidas por Aurora Kade.", "SQL",
     "Consulta tabular con filtro por director."),
    ("¿Cuántas películas hay en la base de datos?", "SQL",
     "Conteo agregado en tablas relacionales."),
    ("Dame el promedio de duración de las películas de la Saga Nova.", "SQL",
     "Agregación numérica sobre campos de la tabla."),
    # --- GRAFO ---
    ("¿Quiénes son los aliados de Captain Stellar?", "GRAFO",
     "La noción de 'aliados' es una relación del grafo."),
    ("¿En qué películas aparecen juntos Captain Stellar y Void Warden?", "GRAFO",
     "Co-aparición/personajes conectados a una misma película."),
    ("¿Cuál es el camino más corto de alianzas entre Aether Lynx y Solar Wisp?", "GRAFO",
     "Consulta de caminos/vecindad en grafo."),
    ("¿Qué personajes tienen más conexiones de enemistad?", "GRAFO",
     "Grado saliente/entrante en relaciones."),
    # --- VECTOR ---
    ("Resume la biografía de Aether Lynx.", "VECTOR",
     "Se necesita texto narrativo desde corpus no estructurado."),
    ("¿De qué trata 'Echoes of the Singularity'?", "VECTOR",
     "Sinopsis/argumento descrito en documentos."),
    ("Explica el conflicto principal de 'Crimson Rift'.", "VECTOR",
     "Análisis temático basado en texto."),
    ("Dame citas relevantes sobre el Orbe Celeste.", "VECTOR",
     "Búsqueda semántica de fragmentos textuales.")
]

INSTRUCCIONES = """
Eres un enrutador de intenciones para un chatbot que dispone de tres fuentes de datos: SQL, GRAFO y VECTOR.
Clasifica la siguiente pregunta del usuario en UNA sola etiqueta: SQL, GRAFO o VECTOR.

Criterios:
- SQL: atributos tabulares (título, año, director, duración, conteos, promedios, filtros simples).
- GRAFO: relaciones entre entidades (alianzas, enemistades, co-apariciones, caminos, centralidad).
- VECTOR: contenido no estructurado (biografías, sinopsis, resúmenes, descripciones libres, citas).

Reglas:
- Si la pregunta implica RELACIONES explícitas entre personajes o nodos, elige GRAFO.
- Si pide atributos directos sin relaciones complejas, elige SQL.
- Si requiere texto narrativo o comprensión semántica amplia, elige VECTOR.
- Devuelve SIEMPRE JSON con el formato:
  {"label":"SQL|GRAFO|VECTOR","justificacion":"<máx 1 frase simple>"}
- No añadas nada más fuera del JSON.

Casos confusos:
- “¿En qué película aparecen juntos X e Y?” → GRAFO.
- “¿Quién dirigió <película>?” → SQL.
- “¿De qué trata <película>?” → VECTOR.
"""

def construir_prompt(pregunta: str) -> str:
    lineas = [INSTRUCCIONES.strip(), "\nEjemplos:"]
    for q, lab, just in FEW_SHOTS:
        lineas.append(f'Usuario: {q}\nSalida: ' + json.dumps({"label": lab, "justificacion": just}, ensure_ascii=False))
    lineas.append(f"\nUsuario: {pregunta}\nSalida:")
    return "\n".join(lineas)

#### Configuración de LLM

In [ ]:
# Configuración del modelo Qwen en Hugging Face
api_key = userdata.get('NLP')
client = InferenceClient(model="Qwen/Qwen2.5-72B-Instruct", token=api_key)

def enrutar_con_qwen(prompt: str, max_new_tokens: int = 64) -> str:
    """
    Genera texto usando el modelo Qwen/Qwen2.5-72B-Instruct.
    """
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-72B-Instruct",
        messages=[
            {"role": "system", "content": "Eres un enrutador de intenciones."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_new_tokens
    )
    return response.choices[0].message["content"].strip()

#### Enrutador principal

In [ ]:
def parsear_salida_llm(texto: str) -> Tuple[Optional[str], Optional[str]]:
    """
    Intenta parsear el JSON {"label":"...","justificacion":"..."}.
    Acepta también salidas mínimas con solo la etiqueta.
    """
    # Intento JSON completo
    try:
        data = json.loads(texto)
        lab = normaliza_etiqueta(data.get("label", ""))
        just = (data.get("justificacion") or "").strip()
        if lab:
            return lab, just
    except Exception:
        pass
    # Intento etiqueta suelta
    lab = normaliza_etiqueta(texto)
    if lab:
        return lab, "Etiqueta simple."
    return None, None

def enrutar_pregunta(pregunta: str) -> RoutingResult:
    """
    Determina a qué fuente (SQL, GRAFO o VECTOR) debe enviarse una
    pregunta, utilizando un modelo de lenguaje para decidir la etiqueta.
    Si no hay consenso, se usa por defecto "VECTOR" como opción más general.
    """
    prompt = construir_prompt(pregunta)
    salida = enrutar_con_qwen(prompt, max_new_tokens=64)

    lab, just = parsear_salida_llm(salida)
    if lab in LABELS:
        return RoutingResult(
            etiqueta=lab,
            justificacion=just
        )

    # Fallback muy conservador → VECTOR
    return RoutingResult(
        etiqueta="VECTOR",
        justificacion="No hubo consenso; elegimos VECTOR por ser la fuente más general.",
    )

#### Pruebas

In [ ]:
pruebas = [
    # SQL
    "¿En qué año se estrenó 'Aegis Rising'?",
    "Lista las películas dirigidas por Aurora Kade.",
    "¿Cuántas películas hay en la base de datos?",
    # GRAFO
    "¿Quiénes son los aliados de Captain Stellar?",
    "¿En qué películas aparecen juntos Captain Stellar y Void Warden?",
    "¿Cuál es el camino más corto de alianzas entre Aether Lynx y Solar Wisp?",
    # VECTOR
    "Resume la biografía de Aether Lynx.",
    "¿De qué trata 'Echoes of the Singularity'?",
    "Dame citas relevantes sobre el Orbe Celeste.",
    # Casos límite
    "¿Qué personajes tienen más conexiones de enemistad?",
    "¿Quién dirigió 'Crimson Rift'?",
    "Explica el conflicto principal de 'Crimson Rift'.",
    "¿Qué duración promedio tienen las películas posteriores a 2018?",
    "¿En qué escenas coaparecen Nova Sable y Iron Scribe?"
]

for p in pruebas:
    res = enrutar_pregunta(p)
    print(f"Q: {p}\n -> etiqueta={res.etiqueta} | justificación: {res.justificacion}\n")

#### Filtrar la fuente

In [ ]:
def decidir_fuente(pregunta: str) -> str:
    """
    Devuelve solo la etiqueta 'SQL' | 'GRAFO' | 'VECTOR' para integrarlo fácilmente
    en tu pipeline (por ejemplo, antes de llamar a tus módulos de consulta).
    """
    return enrutar_pregunta(pregunta).etiqueta

### Implementación de los Módulos de Consulta

##### Módulo SQL

###### Diseño de esquema y prompt

In [ ]:
# Palabras clave prohibidas (solo lectura)
SQL_BANNED = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|ATTACH|DETACH|PRAGMA|REINDEX|VACUUM|TRUNCATE|GRANT|REVOKE)\b",
    re.I
)

def get_sql_schema(db_path=SQLITE_DB_PATH) -> Dict[str, List[Tuple[str, str]]]:
    """
    Devuelve un dict: {tabla: [(columna, tipo_sqlite), ...], ...}
    """
    with sqlite3.connect(str(db_path)) as conn:
        tables = [t[0] for t in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
        ).fetchall()]
        schema = {}
        for t in tables:
            cols = conn.execute(f"PRAGMA table_info({t});").fetchall()
            schema[t] = [(c[1], c[2]) for c in cols]  # (name, type)
    return schema

def sql_schema_text(schema: dict) -> str:
    """
    Texto compacto del esquema para meter al prompt.
    """
    lines = []
    for t, cols in schema.items():
        col_list = ", ".join([f"{c} {tpe}" for c, tpe in cols])
        lines.append(f"- {t}({col_list})")
    return "Esquema SQLite (tablas y columnas):\n" + "\n".join(lines)

def build_sql_prompt(pregunta: str, schema_text: str) -> str:
    """
    Prompt de usuario que verá el modelo, con guía + esquema + pregunta.
    """
    guias = (
        "Eres un generador de SQL para SQLite.\n"
        "- Usa EXCLUSIVAMENTE tablas/columnas del esquema provisto.\n"
        "- SOLO lectura: SELECT o WITH (CTE). Prohibido escribir/modificar.\n"
        "- Si no hay LIMIT, agrega LIMIT 50.\n"
        "- Devuelve ÚNICAMENTE la consulta dentro de ```sql```.\n"
        "- No uses funciones o sintaxis no compatibles con SQLite.\n"
        "- Si el campo es fecha (YYYY-MM-DD) y piden año, usa substr(campo,1,4).\n"
    )
    return f"""{guias}

{schema_text}

Pregunta del usuario:
{pregunta}
"""


###### Limpieza y validación de SQL

In [ ]:
def extract_sql_from_text(texto: str) -> str:
    """
    Extrae el bloque ```sql ... ``` si existe; si no, devuelve el texto completo.
    """
    fence = re.findall(r"```sql\s*(.*?)```", texto, flags=re.I | re.S)
    if fence:
        return fence[0].strip()
    # fallback: intenta sin etiqueta 'sql'
    fence_generic = re.findall(r"```\s*(.*?)```", texto, flags=re.S)
    if fence_generic:
        return fence_generic[0].strip()
    return texto.strip()

def sanitize_sql(sql: str, limit_default: int = 50) -> str:
    """
    - Sin '...'
    - Sin palabras clave prohibidas
    - Asegura LIMIT por defecto
    - Solo SELECT/WITH
    """
    if "..." in sql or "…" in sql:
        raise ValueError("La consulta contiene '...' (incompleta).")

    # Normaliza: quita ';' final para manipular fácilmente
    sql_clean = sql.strip()
    # Evita comentarios maliciosos o segundas sentencias
    # (suficiente para este TP; EXPLAIN/SQLite también ayuda a bloquear)
    if ";" in sql_clean:
        # Permite un único ';' final
        if sql_clean.count(";") > 1 or not sql_clean.endswith(";"):
            raise ValueError("La consulta contiene múltiples sentencias (prohibido).")
        sql_clean = sql_clean.rstrip(";")

    if SQL_BANNED.search(sql_clean):
        raise ValueError("La consulta incluye palabras clave no permitidas.")

    if not re.match(r"^\s*(SELECT|WITH)\b", sql_clean, flags=re.I):
        raise ValueError("La consulta debe iniciar con SELECT o WITH.")

    if not re.search(r"\bLIMIT\b\s+\d+", sql_clean, re.I):
        sql_clean += f"\nLIMIT {limit_default}"

    return sql_clean + ";"

def try_explain(sql: str, db_path=SQLITE_DB_PATH) -> bool:
    """
    Verifica que SQLite pueda planificar la consulta (EXPLAIN QUERY PLAN).
    """
    try:
        with sqlite3.connect(str(db_path)) as conn:
            conn.execute("EXPLAIN QUERY PLAN " + sql)
        return True
    except Exception:
        return False

###### Llamada a LLM

In [ ]:
def generar_sql_con_qwen(prompt: str, max_new_tokens: int = 256) -> str:
    """
    Genera SQL a partir de lenguaje natural, usando Qwen/Qwen2.5-72B-Instruct.
    Devuelve SOLO el texto generado por el modelo (antes de sanitizar).
    """
    # Mensaje de sistema con reglas estrictas
    system_msg = (
        "Eres un generador de consultas SQL para SQLite. "
        "Debes responder SOLO con una consulta válida dentro de un bloque ```sql```; "
        "sin explicaciones, sin comentarios. "
        "Usa únicamente el esquema provisto. NUNCA uses sentencias de escritura o DDL. "
        "Si falta LIMIT, añádelo (LIMIT 50)."
    )

    resp = client.chat.completions.create(
        model="Qwen/Qwen2.5-72B-Instruct",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_new_tokens,
    )

    # Compatibilidad: algunas versiones exponen dict, otras objetos
    msg = resp.choices[0].message
    content = msg["content"] if isinstance(msg, dict) else getattr(msg, "content", "")
    if not content:
        raise RuntimeError("Respuesta vacía del modelo.")
    return content.strip()

def generar_sql(pregunta: str, retries: int = 2, debug: bool = False) -> str:
    """
    Pipeline NL→SQL:
      1) Construye esquema y prompt
      2) Llama a Qwen para generar SQL
      3) Extrae bloque SQL, sanitiza y valida con EXPLAIN
    """
    schema = get_sql_schema(SQLITE_DB_PATH)
    prompt = build_sql_prompt(pregunta, sql_schema_text(schema))

    last_err = None
    for _ in range(max(1, retries)):
        try:
            raw = generar_sql_con_qwen(prompt, max_new_tokens=320)
            if debug:
                print("=== RAW MODEL OUTPUT ===\n", raw)
            sql = extract_sql_from_text(raw)
            sql = sanitize_sql(sql)
            if try_explain(sql):
                return sql
            else:
                last_err = ValueError("El SQL del LLM no pasó EXPLAIN.")
        except Exception as e:
            last_err = e

    # Si llegamos aquí, falló
    raise last_err if last_err else RuntimeError("No se pudo generar SQL válido.")

###### Consultar SQL

In [ ]:
def ejecutar_sql(sql: str, db_path=SQLITE_DB_PATH) -> pd.DataFrame:
    """
    Ejecuta una consulta SQL sobre la base de datos indicada y devuelve
    los resultados en un DataFrame de pandas.
    """
    with sqlite3.connect(str(db_path)) as conn:
        return pd.read_sql_query(sql, conn)

def consultar_sql(pregunta: str):
    """
    Devuelve directamente el resultado crudo de la consulta SQL,
    generado por Qwen 2.5.
    """
    try:
        sql = generar_sql(pregunta, retries=3, debug=False)
        df = ejecutar_sql(sql)
        return df.head(20)
    except Exception as e:
        return str(e)

##### Módulo Grafo

###### Configuraciones básicas

In [ ]:
# Usa la contraseña y el arranque que ya tenés:
NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", INITIAL_PASSWORD)

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    ok = s.run("RETURN 1 AS ok").single()["ok"]
print("Conexión Neo4j OK:", ok == 1)

###### Generador de Cypher

In [ ]:
GRAPH_SCHEMA = textwrap.dedent("""
Modelo de datos (Neo4j):
- Nodos: (:Entidad {name: STRING, tipo: STRING})
- Relaciones: (:Entidad)-[:RELACION {tipo: STRING}]->(:Entidad)
- El campo r.tipo contiene el tipo exacto de relación (ej: ES_HERMANO_DE, ES_ALIADO_DE, ES_ENEMIGO_DE, etc.)

Convenciones para generar Cypher:
- Usa SIEMPRE :Entidad y name para nodos.
- Usa :RELACION y filtra por r.tipo cuando aplique.
- Devuelve columnas con alias legibles.
- No modificar datos, solo consultas MATCH/RETURN/WHERE/ORDER/LIMIT.
""").strip()

SYSTEM_INSTRUCTIONS = (
    "Eres un generador de Cypher. "
    "Devuelve SOLO la consulta Cypher, sin texto adicional ni ```.\n"
    "- Debe comenzar con MATCH o WITH.\n"
    "- Usa la etiqueta :Entidad y la propiedad name.\n"
    "- Filtra por r.tipo cuando corresponda.\n"
    "- Si no hay LIMIT en la respuesta, agrégalo (LIMIT 50)."
)

def _ensure_limit(cypher: str, default_limit: int = 50) -> str:
    """Agrega LIMIT si no existe."""
    if re.search(r"\bLIMIT\s+\d+\b", cypher, flags=re.I):
        return cypher
    return cypher.rstrip(";") + f"\nLIMIT {default_limit}"

def _is_safe_readonly(cypher: str) -> bool:
    """Comprueba que la consulta sea solo de lectura."""
    forbidden = re.compile(
        r"\b(CREATE|DELETE|DETACH|SET|MERGE\s*\(|REMOVE|LOAD\s+CSV|CALL\s+dbms|CALL\s+apoc\.)\b",
        re.I
    )
    return not forbidden.search(cypher)

def _extract_cypher(raw: str) -> str:
    """Extrae la parte Cypher de un texto generado por el modelo."""
    t = raw.strip()
    # Bloque ```cypher
    m = re.search(r"```(?:cypher)?\s*(.*?)```", t, flags=re.S|re.I)
    if m:
        return m.group(1).strip()
    # Buscar desde MATCH/WITH
    m = re.search(r"\b(MATCH|WITH|OPTIONAL MATCH|UNWIND|RETURN)\b", t, flags=re.I)
    return t[m.start():].strip() if m else ""

def generar_cypher_con_qwen(prompt: str, max_new_tokens: int = 64) -> str:
    """Llama a Qwen para generar Cypher."""
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-72B-Instruct",
        messages=[
            {"role": "system", "content": f"{SYSTEM_INSTRUCTIONS}\n\n{GRAPH_SCHEMA}"},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_new_tokens
    )
    return response.choices[0].message.content.strip()

def generate_cypher(question: str) -> str:
    """
    Genera una consulta Cypher de solo lectura a partir de una pregunta natural.
    """
    prompt = f"Pregunta: {question}\nCypher:"
    text = generar_cypher_con_qwen(prompt, max_new_tokens=128)

    cypher = _extract_cypher(text)
    if not cypher or not re.match(r"^\s*(MATCH|WITH)\b", cypher, flags=re.I):
        raise ValueError("El modelo no devolvió un Cypher válido.")

    cypher = _ensure_limit(cypher)

    # Sanitización
    if not _is_safe_readonly(cypher):
        safe_lines = [
            ln for ln in cypher.splitlines()
            if not re.search(r"\b(CREATE|DELETE|DETACH|SET|MERGE\s*\(|REMOVE|LOAD\s+CSV|CALL\s+dbms|CALL\s+apoc\.)\b", ln, re.I)
        ]
        cypher = _ensure_limit("\n".join(safe_lines).strip())
        if not _is_safe_readonly(cypher):
            raise ValueError("El Cypher generado no es de solo lectura.")

    return cypher

###### Consultar Grafo

In [ ]:
def run_cypher(cypher: str):
    """
    Ejecuta una consulta Cypher en la base de datos Neo4j y devuelve
    los resultados tanto en forma de lista de diccionarios como en
    un DataFrame de pandas.
    """
    rows, cols = [], None
    with driver.session() as s:
        result = s.run(cypher)
        cols = result.keys()
        for rec in result:
            rows.append({k: rec.get(k) for k in cols})
    return rows, pd.DataFrame(rows, columns=list(cols) if cols else [])

def consultar_grafo(pregunta: str):
    """
    Devuelve directamente el resultado crudo (lista de filas) de la consulta Cypher,
    generado por la función generate_cypher.
    """
    cypher = generate_cypher(pregunta)
    rows, df = run_cypher(cypher)
    return rows

##### Módulo de Recuperación Híbrida y Re-ranking

###### Búsqueda Vectorial

In [ ]:
# Cache global para BM25
_BM25 = None
_BM25_DOCS = None
_BM25_IDS = None
_CE_MODEL = None

def _tokenize_for_bm25(text: str):
    """
    Tokenización ligera: minúsculas + tokens alfabéticos
    """
    return re.findall(r"[A-Za-zÀ-ÿ0-9_]+", text.lower())

def _ensure_collection():
    """
    Verifica que la colección de Chroma esté disponible.
    Si no existe o no está inicializada, intenta recuperarla
    desde el cliente persistente. Si falla, lanza un error
    indicando que debe ejecutarse la ingesta primero.
    """
    global collection
    if "collection" not in globals() or collection is None:
        # Reutiliza CHROMA_DIR / CHROMA_COLLECTION_NAME si hace falta
        client = chromadb.PersistentClient(path=str(CHROMA_DIR))
        try:
            collection = client.get_collection(CHROMA_COLLECTION_NAME)
        except:
            raise RuntimeError("No se encontró la colección de Chroma. Ejecutá la ingesta primero.")

def search_vector(query: str, k: int = 8):
    """
    Realiza una búsqueda vectorial en la colección usando embeddings
    del texto de la consulta y devuelve los documentos más relevantes
    junto con su metadata e identificadores.
    """
    q_emb = embed_texts([query], batch_size=1, normalize=True)[0]
    res = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=k,
        include=["documents", "metadatas"]
    )
    docs = res["documents"][0]
    metas = res["metadatas"][0]
    ids = res["ids"][0]
    return [{"id": i, "text": d, "meta": m, "source": "vector"} for i, d, m in zip(ids, docs, metas)]

###### Búsqueda por Palabra Clave (BM25)

In [ ]:
def build_bm25_from_chroma(force: bool = False):
    """
    Carga TODOS los documentos de la colección Chroma y crea índice BM25.
    """
    global _BM25, _BM25_DOCS, _BM25_IDS
    _ensure_collection()
    if _BM25 is not None and not force:
        return

    # Traemos todo el corpus (si es grande, ajusta 'limit')
    data = collection.get(include=["documents", "metadatas"], limit=1000000)
    docs = data["documents"]
    ids  = data["ids"]
    # Flatten si collection.get devuelve lista de listas (según versión)
    if docs and isinstance(docs[0], list):
        docs = [d for sub in docs for d in sub]
        ids  = [d for sub in data["ids"] for d in sub]

    tokenized = [_tokenize_for_bm25(t) for t in docs]
    _BM25 = BM25Okapi(tokenized)
    _BM25_DOCS = docs
    _BM25_IDS = ids
    print(f"BM25 listo: {len(_BM25_DOCS)} documentos indexados.")

def search_bm25(query: str, k: int = 8):
    """
    Realiza una búsqueda por palabras clave usando BM25 sobre los documentos
    almacenados en Chroma y devuelve los resultados más relevantes junto con
    su puntaje BM25.
    """
    if _BM25 is None:
        build_bm25_from_chroma()
    scores = _BM25.get_scores(_tokenize_for_bm25(query))
    # Top-k por BM25
    top_idx = np.argsort(scores)[::-1][:k]
    out = []
    for j in top_idx:
        out.append({
            "id": _BM25_IDS[j],
            "text": _BM25_DOCS[j],
            "meta": {},
            "source": "bm25",
            "bm25_score": float(scores[j])
        })
    return out

###### Búsqueda Híbrida

In [ ]:
def hybrid_candidates(query: str, k_vec: int = 8, k_bm25: int = 8):
    """
    Combina resultados de búsqueda vectorial y BM25 para una misma consulta,
    eliminando duplicados por identificador y devolviendo una lista unificada
    de candidatos.
    """
    vec = search_vector(query, k_vec)
    bm  = search_bm25(query, k_bm25)
    by_id = {}
    for it in vec + bm:
        by_id[it["id"]] = it
    return list(by_id.values())

###### Re-ranking

In [ ]:
def get_cross_encoder():
    """
    Obtiene el modelo CrossEncoder para re-ranking.
    Si aún no está cargado, lo inicializa con la versión
    'ms-marco-MiniLM-L-6-v2' y lo guarda en caché global.
    """
    global _CE_MODEL
    if _CE_MODEL is None:
        _CE_MODEL = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)
    return _CE_MODEL

def rerank_with_cross_encoder(query: str, candidates: list, top_n: int = 5):
    """
    Reordena una lista de candidatos usando un modelo CrossEncoder,
    asignando un puntaje de relevancia a cada texto respecto a la consulta.
    Devuelve los mejores resultados según ese puntaje.
    """
    ce = get_cross_encoder()
    pairs = [(query, c["text"]) for c in candidates]
    scores = ce.predict(pairs)
    for c, s in zip(candidates, scores):
        c["ce_score"] = float(s)
    ranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)[:top_n]
    return ranked

def consultar_vector(pregunta: str, k_vec: int = 8, k_bm25: int = 8, top_n: int = 5):
    """
    Devuelve solo el texto del documento más relevante, en una sola línea.
    """
    _ensure_collection()
    cands = hybrid_candidates(pregunta, k_vec, k_bm25)
    ranked = rerank_with_cross_encoder(pregunta, cands, top_n=top_n)

    if not ranked:
        return None

    # Normalizamos: reemplazar saltos de línea múltiples por espacios simples
    texto = ranked[0]["text"]
    texto = " ".join(texto.split())
    return texto

#### Orquestador

In [ ]:
def consultar(pregunta: str, top_n_vector: int = 5):
    """
    Determina la fuente adecuada para responder una pregunta
    (SQL, GRAFO o VECTOR) y delega la consulta a la función
    correspondiente.
    """
    fuente = decidir_fuente(pregunta)  # "SQL" | "GRAFO" | "VECTOR"
    if fuente == "SQL":
        return consultar_sql(pregunta)
    elif fuente == "GRAFO":
        return consultar_grafo(pregunta)
    else:
        return consultar_vector(pregunta, top_n=top_n_vector)

test = [
    # --- Grafo (red de nodos/relaciones) ---
    "¿Quién es hermano de Elara Vance?",                           # Kaelen Vance
    "¿Quién es enemigo de Ironclad?",                               # Silas Vane

    # --- SQL (Peliculas/Personajes/Mundos/Artefactos) ---
    "¿Quién interpreta al personaje Captain Stellar?",             # Brie Larson
    "¿Cuántos minutos dura la película El Fragmento del Eco?",     # 134 minutos

    # --- Vector (corpus de conocimiento) ---
    "¿Cuál es la sinopsis de la película Nexus War: Oblivion?",        # (respuesta: batalla final contra Malakor, el Devorador de Mundos)
    "¿Qué relata la crónica de la Guerra del Nexus?",                  # (respuesta: conflictos que llevaron a la creación de la Alianza Aegis y la amenaza de Malakor)
    ]

for i, q in enumerate(test, start=1):
    print("═" * 60)
    print(f" Q{i}: {q}")
    print("─" * 60)
    res = consultar(q)
    print(f" A{i}: {res}\n")

### Generación de la respuesta

Generar respuesta con LLM

In [ ]:
def generar_respuesta_con_qwen(prompt: str, max_new_tokens: int = 256) -> str:
    """
    Genera texto usando Qwen/Qwen2.5-72B-Instruct vía HF InferenceClient.
    - `prompt` debe incluir la PREGUNTA del usuario y el CONTEXTO/RESULTADOS crudos.
    - El system prompt guía para responder en español, fiel a los datos y sin alucinar.
    """
    system_msg = (
    "Eres el módulo de redacción final de un chatbot multi-fuente.\n"
    "Responde SIEMPRE en español, de forma clara, breve y fiel a los DATOS DEL CONTEXTO.\n"
    "No incluyas mensajes técnicos, logs, porcentajes, ni texto que no sea parte de la respuesta en lenguaje natural."
)

    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-72B-Instruct",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_new_tokens
    )
    return response.choices[0].message["content"].strip()

#### Normalización y armado de prompt

In [ ]:
def _to_serializable(obj: Any) -> Any:
    """Convierte objetos comunes (DataFrame, set, etc.) a algo serializable."""
    try:
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict(orient="records")
    except Exception:
        pass

    if isinstance(obj, (set,)):
        return list(obj)
    if isinstance(obj, (bytes, bytearray)):
        try:
            return obj.decode("utf-8", errors="ignore")
        except Exception:
            return str(obj)
    return obj

def normalizar_resultados(resultados_crudos: Any, max_chars: int = 8000) -> str:
    """
    Acepta string/dict/list/df y devuelve un JSON string compacto, truncado si es muy largo.
    Esto se pasa como CONTEXTO al LLM de redacción.
    """
    try:
        serializable = _to_serializable(resultados_crudos)
        if isinstance(serializable, (dict, list)):
            txt = json.dumps(serializable, ensure_ascii=False, indent=2)
        else:
            txt = str(serializable)
    except Exception:
        txt = str(resultados_crudos)

    # Truncamos para evitar prompts gigantes
    if len(txt) > max_chars:
        txt = txt[:max_chars] + "\n...[TRUNCADO]..."
    return txt

def construir_prompt_respuesta(pregunta: str, resultados_crudos: Any) -> str:
    """
    Arma un prompt estructurado para el generador:
    - Incluye la pregunta del usuario
    - Incluye los resultados crudos (tal como vengan de `consultar`)
    - Indica reglas de fidelidad y estilo
    """
    contexto = normalizar_resultados(resultados_crudos)
    prompt = f"""
TAREA:
Redacta una respuesta breve, clara y fiel a los datos del CONTEXTO. Si el CONTEXTO
no alcanza para responder con certeza, di explícitamente "Información insuficiente".

PREGUNTA:
{pregunta}

CONTEXTO (resultados crudos del sistema de consultas):
{contexto}

REGLAS:
- No inventes nombres, fechas ni relaciones.
- Si hay múltiples filas o items, sintetiza lo esencial.
- Si el resultado proviene de SQL/GRAFO/VECTOR, intégralos coherentemente.
- Responde siempre en español.
"""
    return prompt.strip()

#### Orquestador

In [ ]:
def responder(pregunta: str, *, max_new_tokens: int = 256, debug: bool = True) -> str:
    """
    1) Llama a `consultar(pregunta)` para obtener información cruda.
    2) Construye el prompt final para Qwen con esa info.
    3) Genera la respuesta natural en español.
    """
    # --- 1) Ejecuta tu pipeline existente ---
    try:
        resultados_crudos = consultar(pregunta)
    except NameError:
        raise RuntimeError(
            "No se encontró la función `consultar(pregunta)`. "
            "Defínela en otra celda o impórtala antes de llamar a `responder`."
        )

    # --- 2) Construye el prompt final para redacción ---
    prompt = construir_prompt_respuesta(pregunta, resultados_crudos)

    # --- 3) Genera respuesta con Qwen ---
    try:
        respuesta = generar_respuesta_con_qwen(prompt, max_new_tokens=max_new_tokens)
        return respuesta
    except Exception as e:
        # Fallback minimalista: devuelve un resumen de los datos crudos
        print(f"Falla al invocar Qwen ({type(e).__name__}): {e}")
        resumen_crudo = normalizar_resultados(resultados_crudos, max_chars=1200)
        return (
            "No pude generar la redacción final en este momento.\n\n"
            "A continuación, los datos crudos obtenidos por el sistema:\n"
            f"{resumen_crudo}"
        )

#### Chatbot

In [ ]:
def chatbot():
    """
    Inicia un loop simple de consola:
    - Recibe una pregunta
    - Llama a `responder` (que a su vez usa `consultar` + Qwen)
    - Imprime la respuesta natural
    Escribe 'salir'/'exit'/'quit' para terminar.
    """
    print("💬 Chatbot multi-fuente listo. Escribe tu pregunta (o 'salir' para terminar).")
    while True:
        try:
            pregunta = input("Tú: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Fin del chat.")
            break

        if not pregunta:
            continue
        if pregunta.lower() in {"salir", "exit", "quit"}:
            print("👋 Fin del chat.")
            break

        respuesta = responder(pregunta, max_new_tokens=384, debug=True)
        print(f"\n🤖 Bot: {respuesta}\n")

In [ ]:
chatbot()

## Ejercicio 2: Evolución del Chatbot a un Agente Autónomo

### Abstracción a Herramientas (Tools)

#### Normalización de salidas

In [ ]:
def _to_dataframe(rows_or_df: Any) -> pd.DataFrame:
    """
    Normaliza la salida a DataFrame cuando es posible.
    - Si ya es DataFrame, lo devuelve.
    - Si es lista de dicts o lista de tuplas, intenta convertirlo.
    - Si no es convertible, devuelve DataFrame con una columna 'resultado'.
    """
    if isinstance(rows_or_df, pd.DataFrame):
        return rows_or_df
    try:
        return pd.DataFrame(rows_or_df)
    except Exception:
        return pd.DataFrame({"resultado": [rows_or_df]})

#### Herramientas: SQL, Grafo, Vectorial

In [ ]:
class SQLTool:
    name = "sql"
    description = "Consulta la BD relacional (SQLite) a partir de una pregunta NL."

    def run(self, pregunta: str) -> Dict[str, Any]:
        if "consultar_sql" not in globals() or not callable(globals()["consultar_sql"]):
            raise NotImplementedError("Falta definir consultar_sql(pregunta: str) en tu notebook.")
        try:
            raw = globals()["consultar_sql"](pregunta)  # se espera DataFrame o similar
            df = _to_dataframe(raw)
            return {
                "ok": True,
                "tool": self.name,
                "input": pregunta,
                "preview": df.head(20),
                "dataframe": df,  # útil para agentes que necesiten tabular
            }
        except Exception as e:
            return {"ok": False, "tool": self.name, "input": pregunta, "error": str(e)}

class GraphTool:
    name = "grafo"
    description = "Consulta el grafo de conocimiento (Neo4j) con Cypher a partir de una pregunta NL."

    def run(self, pregunta: str) -> Dict[str, Any]:
        if "consultar_grafo" not in globals() or not callable(globals()["consultar_grafo"]):
            raise NotImplementedError("Falta definir consultar_grafo(pregunta: str) en tu notebook.")
        try:
            raw = globals()["consultar_grafo"](pregunta)  # lista de filas o DataFrame
            df = _to_dataframe(raw)
            return {
                "ok": True,
                "tool": self.name,
                "input": pregunta,
                "preview": df.head(20),
                "dataframe": df,
            }
        except Exception as e:
            return {"ok": False, "tool": self.name, "input": pregunta, "error": str(e)}

class VectorTool:
    name = "vector"
    description = "Recupera y reordena evidencia textual (Chroma + BM25 + CrossEncoder) a partir de una pregunta NL."

    def run(self, pregunta: str) -> Dict[str, Any]:
        if "consultar_vector" not in globals() or not callable(globals()["consultar_vector"]):
            raise NotImplementedError("Falta definir consultar_vector(pregunta: str) en tu notebook.")
        try:
            # Se asume que consultar_vector admite top_n opcional y devuelve estructura usable.
            raw = globals()["consultar_vector"](pregunta, top_n=5)
            # Para vista rápida, intentamos tabular textos/scores si existen.
            if isinstance(raw, dict) and ("docs" in raw or "resultados" in raw):
                docs = raw.get("docs") or raw.get("resultados")
                df = _to_dataframe(docs)
            else:
                df = _to_dataframe(raw)

            return {
                "ok": True,
                "tool": self.name,
                "input": pregunta,
                "preview": df.head(10),
                "dataframe": df,
                "raw": raw,  # se preserva por si el agente necesita más metadatos (scores, ids, etc.)
            }
        except Exception as e:
            return {"ok": False, "tool": self.name, "input": pregunta, "error": str(e)}

#### Integración de las herramientas

In [ ]:
class ToolRegistry:
    def __init__(self, *tools: Any):
        self._by_name = {t.name: t for t in tools}

    def names(self):
        return list(self._by_name.keys())

    def get(self, name: str) -> Any:
        return self._by_name[name]

    def run(self, name: str, pregunta: str) -> Dict[str, Any]:
        tool = self.get(name)
        return tool.run(pregunta)

# Instancia simple de las 3 herramientas:
toolkit = ToolRegistry(SQLTool(), GraphTool(), VectorTool())
print("Herramientas registradas:", toolkit.names())

#### Pruebas

In [ ]:
tests = {
    "grafo": [
        "¿De quién es enemigo 'Ironclad'?",
        "¿Quién es hermano de Kaelen Vance?"
    ],
    "sql": [
        "¿En qué año se estrenó 'Aegis Rising'?",
        "¿Cuál fue la recaudación de 'Nexus War: Oblivion' en dólares?"
    ],
    "vector": [
        "¿De qué trata la película 'Aegis Rising'?",
        "Dime la biografía de Captain Stellar"
    ]
}


for tool_name, preguntas in tests.items():
    for q in preguntas:
        print(f"\n=== Probando {tool_name.upper()} ===")
        out = toolkit.run(tool_name, q)
        if out["ok"]:
            display(out["preview"])
        else:
            print("Error:", out.get("error"))

#### Wrappers LangChain

In [ ]:
def _normalize_preview_to_rows(res: Dict[str, Any]) -> Dict[str, Any]:
    """Convierte DataFrame de 'preview' en lista de dicts para que el LLM lo lea como JSON."""
    payload = {"ok": bool(res.get("ok")), "tool": res.get("tool"), "input": res.get("input")}
    if res.get("ok"):
        df = res.get("preview")
        if isinstance(df, pd.DataFrame):
            payload["rows"] = df.to_dict(orient="records")
        else:
            payload["rows"] = str(df)[:2000]  # fallback textual
    else:
        payload["error"] = res.get("error", "")
    return payload

@tool("bd_router", return_direct=False)
def tool_bd_router(pregunta: str) -> str:
    """
    Enrutador automático a nuestras BD internas usando decidir_fuente(pregunta).
    - Devuelve JSON con {ok, tool, input, rows, route_used}.
    - Solo enruta a: sql | grafo | vector.
    """
    fuente = str(decidir_fuente(pregunta)).strip().lower()
    ruta = fuente if fuente in {"sql", "grafo", "vector"} else "vector"

    res = toolkit.run(ruta, pregunta)
    payload = _normalize_preview_to_rows(res)
    payload["route_used"] = ruta
    return json.dumps(payload, ensure_ascii=False)

### Implementación de Herramientas Externas

#### Herramienta Wikipedia

In [ ]:
# Configura Wikipedia a español por defecto, con fallback a inglés
wikipedia.set_lang("es")

def _wiki_fetch(title: str) -> Dict[str, Any]:
    """
    Obtiene información de Wikipedia para un título dado.
    Devuelve un diccionario con el título real de la página,
    la URL y un resumen truncado.
    Si ocurre un error, devuelve el mensaje de error.
    """
    try:
        page = wikipedia.page(title, auto_suggest=False)
        summary = wikipedia.summary(page.title, sentences=6, auto_suggest=False)
        return {"title": page.title, "url": page.url, "summary": summary[:1200]}
    except Exception as e:
        return {"error": str(e)}

@tool("wikipedia", return_direct=False)
def tool_wikipedia(query: str) -> str:
    """Busca en Wikipedia (es→en fallback) y devuelve JSON con {title, url, summary} del mejor resultado."""
    try:
        hits = wikipedia.search(query)
        if not hits:
            # fallback: intenta en inglés si no hubo hits en español
            wikipedia.set_lang("en")
            hits = wikipedia.search(query)
            wikipedia.set_lang("es")  # restaura
        if not hits:
            return json.dumps({"hits": [], "error": "Sin resultados."}, ensure_ascii=False)
        title = hits[0]
        info = _wiki_fetch(title)
        return json.dumps({"hits": hits[:5], "best": info}, ensure_ascii=False)
    except Exception as e:
        return json.dumps({"error": str(e)}, ensure_ascii=False)

#### Herramienta Web

In [ ]:
# Estado global simple para rate limit entre invocaciones
_WEB_LAST_REQUEST_TIME = 0.0
_WEB_MIN_INTERVAL = 5.0    # segundos (se ajusta dinámicamente ante 429)
_WEB_BASE_SLEEP = 3.0      # para backoff exponencial
_WEB_MAX_SLEEP = 60.0      # tope del backoff
_WEB_BACKENDS = ["lite", "html", "api"]

def _format_query(q: str) -> str:
    ql = q.lower()
    if "sinopsis" in ql:
        return f'"{q}" OR "synopsis" OR "resumen"'
    if "biografía" in ql or "biografia" in ql:
        return f'"{q}" OR "biography" OR "bio"'
    return q

def _wait_for_rate_limit():
    global _WEB_LAST_REQUEST_TIME
    elapsed = time.time() - _WEB_LAST_REQUEST_TIME
    if elapsed < _WEB_MIN_INTERVAL:
        wait_time = _WEB_MIN_INTERVAL - elapsed + random.uniform(1, 3)
        print(f"[web_search] Esperando {wait_time:.1f}s (rate limit)")
        time.sleep(wait_time)

def _exponential_backoff(attempt: int):
    base_wait = _WEB_BASE_SLEEP * (2 ** attempt)
    jitter = random.uniform(0.5, 1.5)
    wait_time = min(base_wait * jitter, _WEB_MAX_SLEEP)
    print(f"[web_search] Backoff intento {attempt+1}: {wait_time:.1f}s")
    time.sleep(wait_time)

def _pick_backend(attempt: int) -> str:
    return _WEB_BACKENDS[attempt % len(_WEB_BACKENDS)]

def _slim_results(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    slim = []
    for r in results or []:
        slim.append({
            "title": r.get("title") or "Sin título",
            "href": r.get("href") or "",
            "body": (r.get("body") or "")[:300]
        })
    return slim

@tool("web_search", return_direct=False)
def tool_web_search(query: str) -> str:
    """
    Búsqueda web con DuckDuckGo (sin API key) con tolerancia a rate limit:
    - Espera mínima entre requests, backoff exponencial con jitter.
    - Cambia de backend entre 'lite'/'html'/'api' si hay problemas.
    - Reformatea la consulta para casos típicos ('sinopsis', 'biografía').
    Devuelve JSON: {"results":[{title, href, body},...], "backend_used", "attempt", "formatted_query"}.
    """
    global _WEB_LAST_REQUEST_TIME, _WEB_MIN_INTERVAL

    formatted = _format_query(query)
    last_err = None

    for attempt in range(7):  # mismo default de retries
        try:
            _wait_for_rate_limit()
            backend = "lite" if attempt == 0 else _pick_backend(attempt)
            # ruido adicional para evitar patrones
            time.sleep(random.uniform(1.5, 4.0))

            _WEB_LAST_REQUEST_TIME = time.time()

            with DDGS() as ddgs:
                results = list(ddgs.text(
                    formatted,
                    max_results=5,
                    safesearch="moderate",
                    region="wt-wt",
                    backend=backend
                ))

                if not results:
                    print("[web_search] Sin resultados con consulta formateada; probando cruda…")
                    results = list(ddgs.text(
                        query,
                        max_results=5,
                        safesearch="moderate",
                        region="wt-wt",
                        backend=backend
                    ))

            slim = _slim_results(results)
            return json.dumps({
                "results": slim,
                "backend_used": backend,
                "attempt": attempt + 1,
                "formatted_query": formatted
            }, ensure_ascii=False)

        except Exception as e:
            last_err = e
            msg = str(e).lower()
            # Heurísticas de error comunes
            if any(tok in msg for tok in ["ratelimit", "429", "status code 202"]):
                print(f"[web_search] Rate limit detectado: {e}")
                _exponential_backoff(attempt)
                _WEB_MIN_INTERVAL = min(_WEB_MIN_INTERVAL * 1.5, 30.0)
            elif "timeout" in msg:
                print(f"[web_search] Timeout: {e}")
                time.sleep(random.uniform(5, 10))
            else:
                print(f"[web_search] Error: {e}")
                time.sleep(random.uniform(2, 6))

    return json.dumps({
        "error": f"No se pudo completar la búsqueda: {last_err}",
        "attempts": 7
    }, ensure_ascii=False)

### Implementación del Agente ReAct

#### Configuración del LLM

In [ ]:
def ask_llm(prompt: str, max_new_tokens: int = 512) -> str:
    """
    Envía un prompt a un modelo de lenguaje y devuelve la respuesta en texto.
    Usa un mensaje de sistema para asegurar claridad y precisión en la salida.
    """
    resp = client.chat.completions.create(
        model=getattr(client, "model", None) or "Qwen/Qwen2.5-72B-Instruct",
        messages=[{"role": "system", "content": "Responde de forma clara y precisa."},
                  {"role": "user", "content": prompt}],
        max_tokens=max_new_tokens
    )
    return resp.choices[0].message["content"].strip()

class LocalChatModel(BaseChatModel):
    """Wrapper simple que adapta `ask_llm` a la interfaz ChatModel de LangChain."""
    model_name: str = "local-llm-adapter"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        sys_msgs = []
        history = []
        for m in messages:
            if isinstance(m, SystemMessage):
                sys_msgs.append(m.content)
            elif isinstance(m, (HumanMessage, AIMessage)):
                history.append(m.content)
            else:
                history.append(str(m.content))

        full_prompt = "\n\n".join(sys_msgs + history)
        content = ask_llm(full_prompt, max_new_tokens=kwargs.get("max_tokens", 512))
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=content))])

    @property
    def _llm_type(self) -> str:
        return "local_chat_wrapper"

#### Construción del agente ReAct

In [ ]:
# LLM (usa tu adaptador; si tienes Qwen ya configurado, se usará automáticamente)
llm = LocalChatModel()

# Herramientas disponibles para el agente:
# - bd_router (usa decidir_fuente -> sql|grafo|vector)
# - wikipedia (solo si el usuario lo pide explícitamente)
# - web_search (solo si lo pide explícitamente)
tools = [tool_bd_router, tool_wikipedia, tool_web_search]

system_instructions = """Eres un agente ReAct. Respondes SIEMPRE en español, con precisión y sin divagar.

POLÍTICA DE HERRAMIENTAS
- Usa SIEMPRE bd_router para consultar nuestras bases internas. bd_router decide automáticamente (sql | grafo | vector) con decidir_fuente(pregunta).
- NO uses wikipedia ni web_search salvo que el usuario lo pida explícitamente (p. ej., “busca en Wikipedia”, “haz una búsqueda web”, “en internet”, “Google”, “DuckDuckGo”).
- Si no hay pedido explícito, limita tus acciones a bd_router.

REGLAS ESTRICTAS DE FORMATO
- NUNCA escribas tú la línea 'Observation:'. El sistema la añadirá después de ejecutar la herramienta.
- NUNCA des 'Final Answer:' sin haber hecho al menos UNA 'Action:' válida (bd_router, wikipedia o web_search) y haber recibido su 'Observation:' del sistema.
- Si no tienes evidencia suficiente, continúa el ciclo Thought -> Action -> Action Input y espera la Observation del sistema. No inventes datos.
- No combines nunca 'Final Answer' y 'Action' en la misma respuesta.
- Si ya diste 'Final Answer:', TERMINA la conversación. No agregues nada más.

FORMATO ReAct (estricto)
Thought: tu razonamiento breve sobre el siguiente paso.
Action: el nombre exacto de una herramienta (bd_router, wikipedia, web_search) si necesitas actuar; si no, omite y pasa a la Final Answer.
Action Input: el input de la herramienta (texto plano).
# (Aquí el SISTEMA insertará) Observation: <salida real de la herramienta>

Cuando tengas suficiente evidencia, finaliza con:
Final Answer: tu respuesta final, concisa y precisa, basada SOLO en las Observations reales (no pegues JSON crudo).

EJEMPLO CANÓNICO
Human: ¿Cuál es la sinopsis breve de 'Aegis Rising'?
Thought: Necesito consultar la base interna para obtener la sinopsis.
Action: bd_router
Action Input: ¿Cuál es la sinopsis breve de 'Aegis Rising'?
# (Sistema) Observation: {{"ok": true, "tool": "sql", "rows": [{{"sinopsis_breve": "El origen de la Alianza Aegis..."}}], "route_used": "sql"}}
Thought: Ya tengo la sinopsis a partir de la Observation.
Final Answer: La sinopsis breve de 'Aegis Rising' es: "El origen de la Alianza Aegis...".
"""

from langchain_core.prompts import ChatPromptTemplate

# IMPORTANTE: incluir {tools} y {tool_names} que LangChain necesita
react_prompt = ChatPromptTemplate.from_messages([
    ("system", system_instructions + "\n\nHerramientas disponibles:\n{tools}\n\nNombres: {tool_names}"),
    ("human", "{input}"),
    # scratchpad como string (evita el error de tipo); el stop_sequence evita que invente Observations
    ("ai", "{agent_scratchpad}"),
])

# Clave: forzar corte en "Observation:" para que el LLM NO la escriba
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt,
    stop_sequence="Observation:",
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,                 # imprime Thought/Action/Observation
    handle_parsing_errors=True,
    return_intermediate_steps=True,
    max_iterations=4,
)

def run_agent_checked(pregunta: str):
    """Ejecuta el agente y muestra qué tools usó realmente."""
    result = agent_executor.invoke({"input": pregunta})
    steps = result.get("intermediate_steps", [])
    used = [a.tool for (a, _) in steps]
    print("\nTOOLS_USED:", used)
    if not any(t in ("bd_router", "sql", "grafo", "vector", "wikipedia", "web_search") for t in used):
        print("⚠️ El agente NO ejecutó ninguna herramienta. (Posible alucinación)")
    print("\n" + "="*80)
    print("RESPUESTA FINAL:\n", result["output"])
    print("="*80)
    return result

print("Agente ReAct listo. Herramientas:", [t.name for t in tools])


#### Prueba

In [ ]:
run_agent_checked("¿Quién es el enemigo de Ironclad? Luego busca en Wikipedia el género de la película?")

In [ ]:
run_agent_checked("Dime la sinopsis de Aegis Rising. Además cuéntame ¿En qué año se estrenó la película?")

In [ ]:
run_agent_checked("¿De quién es la identidad Ironclad? También busca en Wikipedia qué narra la película")

In [ ]:
run_agent_checked("¿Quién es mentor de Captain Stellar? Además cuéntame su biografía")